# ElGamal Broken Proof Repair Experiment

This notebook runs the `elgamal-broken-repair` experiment with:
- **Shortest-first ordering** — trials proceed from simplest to hardest proof
- **Adaptive step limits** — each trial gets `1.4x` the number of tactic lines in the broken proof
- **Improved prompt** — the broken proof is presented as the primary strategy to follow closely
- **Judgment type hints** — the agent is told whether it's in hoare/phoare/equiv context

Prior runs did not have the above features and had a success rate of 1/4.
- The agent was not carefully following the prompts.

## Prerequisites
- `DEEPSEEK_API_KEY` set in environment
- LM Studio running locally (for embeddings)
- EasyCrypt binary available at `integration/extern/easycrypt/_build/default/src/ec.exe`


In [1]:
import os
import sys
from pathlib import Path

# Ensure the project root is on the path
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"DEEPSEEK_API_KEY set: {'DEEPSEEK_API_KEY' in os.environ}")

Project root: /Users/k323lee/git/AI4EC
DEEPSEEK_API_KEY set: False


## Configuration

In [2]:
from dataclasses import replace as dc_replace

from integration.agent.config import AgentConfig, apply_deepseek_provider
from integration.experiment.config import ExperimentConfig
from integration.experiment.corpora.elgamal import ElGamalCorpus
from integration.experiment.protocols import ExperimentSpec, BrokenFormalConfig
from integration.experiment.runner import run_experiment, run_broken_formal_trial
from integration.experiment.proof_extract import apply_lines, strip_tactics

# --- Experiment parameters (adjust these) ---
DEEPSEEK_MODEL = "deepseek-v4-pro"       # or "deepseek-v4-pro"
THINKING_MODE = "adaptive"                  # disabled | enabled | adaptive
REASONING_EFFORT = None                     # None | "high" | "max"
EMBED_MODEL = "text-embedding-nomic-embed-text-v1.5"
MAX_TRIALS = 10                             # how many proofs to attempt (shortest first)
ADAPTIVE_MULTIPLIER = 1.4                   # step budget = 1.4x proof lines
MIN_STEPS = 10                              # floor for step budget
STUCK_LIMIT = 20                            # stuck counter before giving up
TOP_K_PREMISES = 10                         # premises in prompt
LLM_MAX_TOKENS = 16384                      # max output tokens per completion
COST_LIMIT_USD = 1.00                       # stop experiment after spending this much

DATA_DIR = Path("data")
OUTPUT_DIR = None  # None = auto-timestamped under integration/output/experiments/

## Build Experiment Spec and Config

In [3]:
# Build agent config
agent = AgentConfig(
    top_k=TOP_K_PREMISES,
    llm_max_tokens=LLM_MAX_TOKENS,
    embed_model=EMBED_MODEL,
)
apply_deepseek_provider(
    agent,
    model=DEEPSEEK_MODEL,
    thinking=THINKING_MODE,
    reasoning_effort=REASONING_EFFORT,
)

# Build experiment config with new features
exp_config = ExperimentConfig(
    spec_name="elgamal-broken-repair",
    trials=MAX_TRIALS,
    stuck_limit=STUCK_LIMIT,
    data_dir=DATA_DIR,
    agent=agent,
    sort_by_difficulty=True,              # shortest proof first
    adaptive_steps_multiplier=ADAPTIVE_MULTIPLIER,
    min_adaptive_steps=MIN_STEPS,
    cost_limit_usd=COST_LIMIT_USD,
)
if OUTPUT_DIR is not None:
    exp_config.output_dir = Path(OUTPUT_DIR)

print(f"Output dir: {exp_config.output_dir}")
print(f"Sort by difficulty: {exp_config.sort_by_difficulty}")
print(f"Adaptive steps: {exp_config.adaptive_steps_multiplier}x (min {exp_config.min_adaptive_steps})")

Output dir: /Users/k323lee/git/AI4EC/integration/output/experiments/run-20260728T160129Z
Sort by difficulty: True
Adaptive steps: 1.4x (min 10)


## Preview: Available Proof Cases (sorted by difficulty)

In [ ]:
# Load and preview all available cases
corpus = ElGamalCorpus(data_dir=DATA_DIR, sandbox_dir=exp_config.output_dir / "sandboxes")
all_cases = corpus.load_cases()
all_cases.sort(key=lambda c: len(c.tactic_lines))

print(f"Total available proofs: {len(all_cases)}")
print(f"Will attempt: {min(MAX_TRIALS, len(all_cases))} (shortest first)")
print()
print(f"{'#':<3} {'Name':<20} {'Lines':<7} {'Adaptive Steps':<15}")
print("-" * 50)
for i, case in enumerate(all_cases):
    n_lines = len(case.tactic_lines)
    steps = max(MIN_STEPS, int(ADAPTIVE_MULTIPLIER * n_lines))
    marker = " <--" if i < MAX_TRIALS else ""
    print(f"{i:<3} {case.name:<20} {n_lines:<7} {steps:<15}{marker}")

## Run the Experiment

Runs trials shortest-first, stopping early if `COST_LIMIT_USD` is exceeded.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Build the spec with sandbox dir
spec = ExperimentSpec(
    name="elgamal-broken-repair",
    corpus=ElGamalCorpus(data_dir=DATA_DIR, sandbox_dir=exp_config.output_dir / "sandboxes"),
    broken_formal=BrokenFormalConfig(),
)

# Run — stops automatically when COST_LIMIT_USD is reached
result = run_experiment(spec, exp_config)

## Results Summary

In [ ]:
print(f"Trials run: {result.trials_run}, skipped: {result.trials_skipped}")
print(f"Successes: {result.successes}")
print(f"Stuck: {result.stuck}")
print(f"Max steps reached: {result.max_steps}")
print(f"Errors: {result.errors}")
print()
if result.estimated_cost:
    print(f"Total cost: ${result.estimated_cost['usd']:.4f} USD ({result.estimated_cost['model']})")
    print(f"Cost limit: ${COST_LIMIT_USD:.2f}")

usage = result.token_usage
print(f"Total tokens: {usage.total_tokens:,} ({usage.calls} API calls)")
print(f"  Input: {usage.prompt_tokens:,} (cache hit: {usage.cached_prompt_tokens:,})")
print(f"  Output: {usage.completion_tokens:,} (reasoning: {usage.reasoning_tokens:,})")

## Per-Trial Breakdown

In [ ]:
import pandas as pd

rows = []
for tr in result.trial_results:
    rows.append({
        "trial": tr.trial_id,
        "name": tr.name,
        "reason": tr.reason,
        "steps": tr.steps,
        "duration_s": tr.duration_s,
        "cost_usd": tr.estimated_cost["usd"] if tr.estimated_cost else None,
        "skipped": tr.skipped if hasattr(tr, 'skipped') else False,
    })

df = pd.DataFrame(rows)
df

In [ ]:
# Visualize success/failure by proof complexity
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

colors = {
    "COMPLETE": "green",
    "MAX_STEPS": "orange",
    "STUCK": "red",
    "SKIPPED": "gray",
}

for _, row in df.iterrows():
    color = colors.get(row["reason"], "blue")
    ax.barh(row["name"], row["steps"], color=color, alpha=0.7)

ax.set_xlabel("Steps taken")
ax.set_title("ElGamal Broken Repair: Steps by Trial (sorted by difficulty)")

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, alpha=0.7, label=k) for k, c in colors.items()]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.show()

## Inspect Failed Trials

Look at the retrospective for failed trials to understand what went wrong.

In [ ]:
import json

failed = df[df["reason"].isin(["MAX_STEPS", "STUCK"])]
print(f"{len(failed)} failed trials\n")

for _, row in failed.iterrows():
    tr = result.trial_results[row["trial"]]
    retro_path = tr.retrospective_file
    if retro_path and Path(retro_path).exists():
        retro = json.loads(Path(retro_path).read_text())
        print(f"--- Trial {row['trial']}: {row['name']} ({row['reason']}) ---")
        if "response" in retro and retro["response"]:
            resp = retro["response"]
            if isinstance(resp, dict):
                reason = resp.get("failure_reason") or resp.get("why_failed") or resp.get("root_cause", "")
                print(f"  Failure reason: {reason[:200]}")
            else:
                print(f"  Response: {str(resp)[:200]}")
        print()

## Single-Trial Debug Mode

Run a single trial interactively for detailed debugging.

In [ ]:
# Pick a specific case to debug (0 = shortest/easiest)
CASE_INDEX = 0

case = all_cases[CASE_INDEX]
n_lines = len(case.tactic_lines)
adaptive_steps = max(MIN_STEPS, int(ADAPTIVE_MULTIPLIER * n_lines))

print(f"Case: {case.name}")
print(f"Proof lines: {n_lines}")
print(f"Adaptive max_steps: {adaptive_steps}")
print(f"Source: {case.file}")
print()

# Show the broken proof text
lines = case.file.read_text(encoding='utf-8').splitlines()
broken_text = '\n'.join(lines[i-1] for i in case.tactic_lines)
print("Broken proof:")
print(broken_text)

In [ ]:
# Run that single trial
from dataclasses import replace
from integration.experiment.runner import run_broken_formal_trial

single_trial_dir = exp_config.output_dir / "debug_trials" / f"debug_{case.name}"

# Override agent max_steps for this trial
debug_config = replace(
    exp_config,
    adaptive_steps_multiplier=ADAPTIVE_MULTIPLIER,
    min_adaptive_steps=MIN_STEPS,
)

trial_result = run_broken_formal_trial(
    trial_id=0,
    case=case,
    config=debug_config,
    trial_dir=single_trial_dir,
)

print(f"Result: {trial_result.reason}")
print(f"Steps: {trial_result.steps}")
print(f"Message: {trial_result.message}")
print(f"Duration: {trial_result.duration_s:.1f}s")
if trial_result.estimated_cost:
    print(f"Cost: ${trial_result.estimated_cost['usd']:.4f}")

In [ ]:
# Inspect the agent log for the debug trial
log_path = single_trial_dir / "agent_log.json"
if log_path.exists():
    log_data = json.loads(log_path.read_text())
    steps = log_data.get("steps", [])
    print(f"Total steps recorded: {len(steps)}")
    print()
    for step in steps[-5:]:  # Show last 5 steps
        print(f"  Step {step.get('step', '?')}: {step.get('action', '?')} -> {step.get('outcome', '?')}")
        if step.get('error'):
            print(f"    Error: {step['error'][:100]}")